# 📊 Rates, not counts — and your first pandas figure

**Time:** ~40 minutes &nbsp;•&nbsp; **Goal:** learn the one habit that separates an analysis from a spreadsheet, and draw it.

In the last notebook you ranked crops by **bumblebee visits per watch**. Perfectly reasonable. In this notebook you will rank them by **visits per flower**, and get a substantially different answer.

Neither ranking is a mistake. They answer different questions, and the difference is the most important idea in this lab:

> A count measures the insects **and the effort**. A rate separates them.

Every survey, every screen, every experiment with unequal groups has this problem. You will meet it for the rest of your career.

## Run this first

In [ ]:
# Run this first. Everything sessions 1 and 2 taught you, in one cell.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

df.columns = df.columns.str.strip()                      # the header bug
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
df["Date"] = pd.to_datetime(df["Date"])                  # real dates
df["Month"] = pd.Categorical(                            # months in season order
    df["Date"].dt.month_name(),
    categories=["April", "May", "June", "July", "August", "September"],
    ordered=True,
)
df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]
df["All_Insects"] = df[INSECTS].sum(axis=1)

import matplotlib.pyplot as plt

print(df.shape)
df.head(3)

## Skill 1 — The reversal

Start with the cleanest case in the dataset: community plots versus individual plots.

Ask it with a mean count, and community plots win. Ask it per flower, and individual plots win. Same 1,620 rows.

In [ ]:
# DEMO - ask the same question two ways
g = df.groupby("Community").agg(
    watches=("All_Insects", "size"),
    insects=("All_Insects", "sum"),
    flowers=("Flowers", "sum"),
)

g["mean_per_watch"] = g["insects"] / g["watches"]
g["per_flower"] = g["insects"] / g["flowers"]

g.round(4)

Read the two right-hand columns against each other.

- **Per watch**, community plots see more insects — 1.37 against 1.08.
- **Per flower**, individual plots see more — 0.0142 against 0.0102.

The resolution is in the `flowers` column. Community plots put up far more flowers per watch, so each individual bloom gets *less* attention even though the plot as a whole gets more. Whether that is good news depends entirely on whether you are the allotment holder or the flower.

A count answers "where were the insects?". A rate answers "how attractive was this?". Decide which you meant **before** you run the groupby.

### 📊 DIY challenge

1. Do the same comparison for `Year`. Were 2018's extra insects real, or extra surveying? (2018 has 1,139 watches to 2017's 481.)
2. Do it for `Bumblebees` alone instead of all insects. Does the reversal still hold?
3. Add a column giving flowers per watch for each group. Does it explain the direction of the flip?

> 💡 Build the rate from **two sums**, not from a mean of per-row rates. Those are different numbers, and the next skill explains why.

In [ ]:
# TODO - your turn

## Skill 2 — Two rates, and the one you want

There are two ways to compute "visits per flower" for a group, and they do not agree:

```python
# A: pooled - add up all visits, add up all flowers, divide once
g["insects"].sum() / g["flowers"].sum()

# B: mean of per-row rates - divide each row, then average
(df["All_Insects"] / df["Flowers"]).mean()
```

**A weights every flower equally. B weights every watch equally.**

B is dominated by the small watches: a single bee on a plot with one open courgette flower scores 1.0, and that one row pulls the average up as hard as a 2,000-flower raspberry patch pulls it down. On this dataset the two differ by a factor of ten.

Neither is wrong, but A is what "visits per flower" means in plain English, and it is what you want here.

In [ ]:
# DEMO - the two rates, side by side
df["Row_Rate"] = df["All_Insects"] / df["Flowers"]

compare = df.groupby("Community").agg(
    pooled=("All_Insects", "sum"),
    flowers=("Flowers", "sum"),
    mean_of_rates=("Row_Rate", "mean"),
)
compare["pooled_rate"] = compare["pooled"] / compare["flowers"]

print(compare[["pooled_rate", "mean_of_rates"]].round(4))

print()
print("smallest watches, where mean-of-rates gets its big numbers:")
print(df.nlargest(4, "Row_Rate")[["Crop", "Flowers", "All_Insects", "Row_Rate"]])

### 📊 DIY challenge

1. How many watches had **5 or fewer** flowers in view? What is the mean `Row_Rate` for just those?
2. Recompute `mean_of_rates` after dropping watches with fewer than 10 flowers. Does it move towards the pooled rate?
3. Which of the two would you report, and what one sentence would you write to justify it?

> 💡 Whenever you divide small numbers by small numbers, check how small. A ratio with a denominator of 1 is not a measurement.

In [ ]:
# TODO - your turn

## Skill 3 — Your first figure from a groupby

You met `fig, ax = plt.subplots()` in the lecture. Everything from there is the same — the only new thing is that a pandas object can draw **itself** onto your `ax`:

```python
fig, ax = plt.subplots()
series.plot(kind="barh", ax=ax)
ax.set_xlabel("visits per flower")
```

Two things to hold on to:

- `ax=ax` is not optional. Leave it out and pandas makes its own figure, and every `ax.set_...` line you write afterwards decorates the wrong one.
- `.plot()` uses the **index** for the labels and the **values** for the bars. Which means all the work is in getting the groupby right; the plotting is one line.

Since the index does the labelling, sort before you plot. A sorted bar chart is readable; an unsorted one is a puzzle.

In [ ]:
# DEMO - bumblebee visits per flower, by crop
crops = df.groupby("Crop").agg(
    watches=("Bumblebees", "size"),
    bumblebees=("Bumblebees", "sum"),
    flowers=("Flowers", "sum"),
)
crops = crops[crops["watches"] >= 40]              # the n-guard from notebook 5
crops["per_flower"] = crops["bumblebees"] / crops["flowers"]

fig, ax = plt.subplots()
fig.set_size_inches(7, 4)

crops["per_flower"].sort_values().plot(kind="barh", ax=ax, color="#5b9df9")

ax.set_xlabel("bumblebee visits per flower")
ax.set_ylabel("")
ax.set_title("Which crops do bumblebees actually work?")
plt.show()

### 📊 DIY challenge

1. Redraw that chart for **honeybees**. Which crop moves the most?
2. Add the sample size to the labels so a reader can judge them. One way:
   `crops.index = crops.index + " (n=" + crops["watches"].astype(str) + ")"`
3. Try it without `ax=ax` and see what happens to your title. Now you will remember.

> 💡 `kind=` takes `"bar"`, `"barh"`, `"line"`, `"scatter"`, `"hist"`, `"box"`. It is a shortcut to the `ax.` methods you already know, not a replacement for them.

In [ ]:
# TODO - your turn

## Skill 4 — Draw the reversal

A figure argues better than a table. Put the two rankings side by side and the point makes itself — this is exactly the `plt.subplots(1, 2)` from the lecture, with a groupby feeding each panel.

In [ ]:
# DEMO - the same crops, ranked two ways
crops["per_watch"] = crops["bumblebees"] / crops["watches"]

fig, axes = plt.subplots(1, 2)
fig.set_size_inches(11, 4)

crops["per_watch"].sort_values().plot(kind="barh", ax=axes[0], color="#d6a44c")
axes[0].set_title("Bumblebees per watch")
axes[0].set_xlabel("visits per watch")

crops["per_flower"].sort_values().plot(kind="barh", ax=axes[1], color="#5b9df9")
axes[1].set_title("Bumblebees per flower")
axes[1].set_xlabel("visits per flower")

fig.tight_layout()
plt.show()

Find **Squash** and **Courgette** in both panels. Near the bottom on the left, near the top on the right.

Squash and courgette produce a handful of enormous flowers — 1,436 flowers across 220 watches, about six per watch, against raspberry's 28,403. So a watch rarely sees a bee, but each individual bloom is worked hard. On the left you are measuring how many flowers the plant puts up. On the right you are measuring how attractive they are.

`fig.tight_layout()` is worth remembering too: it stops the left panel's labels from being eaten by the right panel.

### 📊 DIY challenge

1. Make the same two-panel figure for honeybees. Does squash move the same way?
2. Put bumblebees in the left panel and honeybees in the right, both per flower. Which crops separate the two?
3. Give the whole figure one heading with `fig.suptitle(...)`, and make sure `tight_layout()` still leaves room for it.

> 💡 `axes` is an array. `axes[0]` is the left panel. With `plt.subplots(2, 2)` it becomes 2-D and you index it `axes[0, 1]`.

In [ ]:
# TODO - your turn

## Skill 5 — The shape of a season 🏆

A grouped **line** needs the x-axis in the right order, which is why you made `Month` an ordered Categorical back in session 2. Now it pays off: `groupby` respects that order, and the line comes out in calendar sequence without any further work.

In [ ]:
# DEMO - two pollinators through the summer
season = df.groupby("Month", observed=True).agg(
    watches=("Flowers", "size"),
    flowers=("Flowers", "sum"),
    bumblebees=("Bumblebees", "sum"),
    honeybees=("Honeybees", "sum"),
)
season["bb_rate"] = season["bumblebees"] / season["flowers"]
season["hb_rate"] = season["honeybees"] / season["flowers"]

fig, ax = plt.subplots()
fig.set_size_inches(7, 4)

season["bb_rate"].plot(ax=ax, marker="o", label="bumblebees", color="#5b9df9")
season["hb_rate"].plot(ax=ax, marker="o", label="honeybees", color="#d6a44c")

ax.set_ylabel("visits per flower")
ax.set_xlabel("")
ax.set_title("Pollinator activity through the season")
ax.legend()
plt.show()

season.round(4)

Both rates climb steadily from April to August. Now look back at the `flowers` column: April had 25,911 flowers in view and August only 22,796, on twice as many watches. April is all fruit blossom — vast numbers of flowers, very few insects out yet.

Plot the raw counts instead and you would see a much flatter, duller picture. The rate is what makes the season visible.

### 🏆 DIY finale — one figure that answers a question

Pick **one** question and build a single, finished figure that answers it. Suggestions:

- Which allotment is best for bumblebees, per flower? (n-guard: at least 100 watches.)
- Do community and individual plots differ across the season? (Group by `Month` **and** `Community`, then `.unstack()` — a two-column frame plots as two lines automatically.)
- Which bumblebee species dominates, and does that change between months?
- Is 2018 really better than 2017, once you divide by flowers?

Your figure must have:

1. axis labels with units — "visits per flower", not "rate"
2. a title that states the finding, not the variables
3. sample sizes visible somewhere, or a stated n-guard
4. sorted bars, or an axis in a sensible order

Then write two sentences underneath: what the figure shows, and one reason it might be misleading.

> 💡 The second sentence is the hard one and the valuable one. Every figure in this notebook has an answer to it.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- pandas' own plotting guide — all the `kind=` options with pictures: <https://pandas.pydata.org/docs/user_guide/visualization.html>
- Simpson's paradox, which is the formal name for what Skill 1 showed you: <https://en.wikipedia.org/wiki/Simpson%27s_paradox>
- `seaborn` sits on top of matplotlib and is very good at grouped data. Try `import seaborn as sns; sns.barplot(data=df, x="Community", y="Row_Rate")` and notice it draws error bars for free.

## Recap

You can now:

- see the difference between a **count** and a **rate**, and say which question each answers
- choose between a pooled rate and a mean of per-row rates, and justify it
- draw a groupby result with `series.plot(kind="barh", ax=ax)` — and remember `ax=ax`
- put two rankings side by side with `plt.subplots(1, 2)` to show a reversal
- get a seasonal line in calendar order, because the month column knows its own order

**Next (optional):** `07_reshape_and_merge` — the two operations that turn a wide survey sheet into a table you can group by anything, and let you bring in information the file does not contain.